# PitchTrack — extract the fundamental frequency (f0)

The pitch track has several stages; this notebook covers the **first stage**:
extracting the fundamental frequency `f0` (and the sound-pressure level `SPL`)
per analysis frame with `comsar.PitchTrack`, saving it as CSV, and viewing it in
an interactive **pitch player**. Later stages (melody / note segmentation, tonal
system) build on this f0 track.

> Requires `bader-comsar` plus `soundfile`.

In [ ]:
from comsar import PitchTrack, pitch_player

## Analysis windows

Windowing is given in time units (milliseconds + overlap fraction), so any
sample rate works. 25 ms windows with 60 % overlap correspond to the historical
1102-sample window with a 441-sample hop at 44.1 kHz.

In [ ]:
tt = PitchTrack(window_ms=25.0, overlap=0.6)

## Extract f0 from a single recording

`extract` returns a `TrackResult`; `.features` is a DataFrame indexed by
`time_s` with the columns `Pitch` (f0 in Hz) and `SPL`. Unvoiced / silent
frames have `Pitch = 0`.

In [ ]:
WAV = "CAM-5_SARA PANH.wav"
result = tt.extract(WAV)
result.features.head()

## Save f0 as CSV

Written next to the audio file. `EXCEL_STYLE` uses semicolons + decimal commas
for European Excel (set to `False` for a standard comma CSV; read the Excel
style back with `pd.read_csv(path, sep=";", decimal=",", index_col=0)`).

In [ ]:
EXCEL_STYLE = True
sep, dec = (";", ",") if EXCEL_STYLE else (",", ".")

CSV_FILE = WAV.rsplit(".", 1)[0] + "_f0.csv"
result.features.to_csv(CSV_FILE, sep=sep, decimal=dec)
print(f"saved {len(result.features)} frames to {CSV_FILE}")

## Pitch player

The player shows the grey waveform on top and the f0 track on a logarithmic
frequency axis below. Press **Play** to listen; a vertical cursor follows the
playback position, and clicking the plot seeks to that point. Unvoiced frames
leave a gap in the line. Further pitch-track layers (melody, tonal system) will
be added to this player in later stages.

In [ ]:
pitch_player(WAV, result)